# Supplementary Naive Bayes, RNN, and LSTM 5-Fold CV

This supplementary exploratory analysis follows the leakage-safe methodology of the primary corrected 5-fold notebook. It uses the original real dataset, fold-local predictor normalization, fresh fold-specific CTGAN training on fold-training minority rows only, and validation only on untouched real fold observations.

The RNN and LSTM treat tabular predictors as an artificial feature sequence; the data do not contain natural temporal observations. These three models are supplementary and do not replace the six primary models or the official sensitivity analysis.


In [1]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_PATH = r'../data/Final_Encoded.csv'
FOLD_RESULTS_PATH = r'../results/additional_models_5fold_fold_results.csv'
SUMMARY_PATH = r'../results/additional_models_5fold_summary.csv'
TARGET_COLUMN = 'CGPA3_Class'
real_df = pd.read_csv(DATA_PATH)
class_counts = real_df[TARGET_COLUMN].value_counts()
MINORITY_CLASS = class_counts.idxmin()
feature_columns = [c for c in real_df.columns if c not in [TARGET_COLUMN, 'Target', 'Current_CGPA5']]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Target values:', sorted(real_df[TARGET_COLUMN].unique().tolist()))
print('Class counts:', class_counts.sort_index().to_dict())
print('Minority class:', MINORITY_CLASS)


Target values: [0, 1, 2]
Class counts: {0: 655, 1: 608, 2: 218}
Minority class: 2


In [2]:
class ExploratorySequenceModel(nn.Module):
    def __init__(self, model_type, hidden_size=32, num_classes=3):
        super().__init__()
        recurrent = nn.RNN if model_type == 'RNN' else nn.LSTM
        self.recurrent = recurrent(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, num_classes)

    def forward(self, inputs):
        sequence_output, _ = self.recurrent(inputs)
        return self.output(sequence_output[:, -1, :])

def record_result(results, fold, model, y_true, predictions, probabilities):
    results.append({
        'Fold': fold, 'Model': model,
        'Accuracy': accuracy_score(y_true, predictions),
        'Precision': precision_score(y_true, predictions, average='macro'),
        'Recall': recall_score(y_true, predictions, average='macro'),
        'F1': f1_score(y_true, predictions, average='macro'),
        'ROC-AUC': roc_auc_score(y_true, probabilities, multi_class='ovr', average='macro')
    })

fold_results = []
for fold, (train_idx, validation_idx) in enumerate(cv.split(real_df, real_df[TARGET_COLUMN]), start=1):
    fold_train = real_df.iloc[train_idx].copy()
    fold_validation = real_df.iloc[validation_idx].copy()

    predictor_columns = [c for c in fold_train.columns if c != TARGET_COLUMN]
    train_max = fold_train[predictor_columns].max()
    normalized_train = fold_train.copy()
    normalized_validation = fold_validation.copy()
    for column in predictor_columns:
        if train_max[column] > 0:
            normalized_train[column] = normalized_train[column] / train_max[column]
            normalized_validation[column] = normalized_validation[column] / train_max[column]

    minority_train = normalized_train[normalized_train[TARGET_COLUMN] == MINORITY_CLASS].copy()
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(minority_train)
    for column in minority_train.columns:
        metadata.update_column(column, sdtype='numerical')
    synthesizer = CTGANSynthesizer(metadata, epochs=500, batch_size=50, generator_dim=(256, 256), discriminator_dim=(256, 256), generator_lr=2e-4, discriminator_lr=2e-4, verbose=False)
    synthesizer.fit(minority_train)

    majority_count = normalized_train[TARGET_COLUMN].value_counts().max()
    synthetic_count = int(majority_count - len(minority_train))
    synthetic_train = synthesizer.sample(num_rows=synthetic_count)
    for column in predictor_columns:
        synthetic_train[column] = synthetic_train[column].clip(0, 1)
    synthetic_train[TARGET_COLUMN] = MINORITY_CLASS
    augmented_train = pd.concat([normalized_train, synthetic_train], ignore_index=True)

    X_train = augmented_train[feature_columns].values
    y_train = augmented_train[TARGET_COLUMN].astype(int).values
    X_validation = normalized_validation[feature_columns].values
    y_validation = normalized_validation[TARGET_COLUMN].astype(int).values

    nb = GaussianNB()
    nb.fit(X_train, y_train)
    record_result(fold_results, fold, 'Naive Bayes', y_validation, nb.predict(X_validation), nb.predict_proba(X_validation))

    sequence_train = torch.FloatTensor(X_train).unsqueeze(-1)
    sequence_validation = torch.FloatTensor(X_validation).unsqueeze(-1)
    loader = DataLoader(TensorDataset(sequence_train, torch.LongTensor(y_train)), batch_size=32, shuffle=True)
    for model_name in ['RNN', 'LSTM']:
        torch.manual_seed(SEED + fold)
        model = ExploratorySequenceModel(model_name)
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        model.train()
        for _ in range(100):
            for batch_x, batch_y in loader:
                optimizer.zero_grad()
                loss = criterion(model(batch_x), batch_y)
                loss.backward()
                optimizer.step()
        model.eval()
        with torch.no_grad():
            outputs = model(sequence_validation)
            probabilities = torch.softmax(outputs, dim=1).numpy()
            predictions = outputs.argmax(dim=1).numpy()
        record_result(fold_results, fold, model_name, y_validation, predictions, probabilities)

fold_results_df = pd.DataFrame(fold_results)
print(fold_results_df.to_string(index=False))


c:\Users\arsayeem\AppData\Local\Programs\Python\Python310\lib\site-packages\sdv\single_table\base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
c:\Users\arsayeem\AppData\Local\Programs\Python\Python310\lib\site-packages\sdv\single_table\base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\arsayeem\AppData\Local\Programs\Python\Python310\lib\site-packages\sdv\single_table\base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
c:\Users\arsayeem\AppData\Local\Programs\Python\Python310\lib\site-packages\sdv\single_table\base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warni

 Fold       Model  Accuracy  Precision   Recall       F1  ROC-AUC
    1 Naive Bayes  0.410774   0.412540 0.464259 0.400106 0.604858
    1         RNN  0.420875   0.414777 0.466074 0.416317 0.610826
    1        LSTM  0.414141   0.397710 0.417768 0.397383 0.615000
    2 Naive Bayes  0.432432   0.418665 0.456406 0.393704 0.588593
    2         RNN  0.429054   0.414914 0.422944 0.417622 0.593772
    2        LSTM  0.385135   0.387431 0.397979 0.352166 0.580928
    3 Naive Bayes  0.385135   0.397202 0.422987 0.358460 0.576468
    3         RNN  0.425676   0.411000 0.440103 0.413574 0.594373
    3        LSTM  0.375000   0.353434 0.380118 0.339685 0.558395
    4 Naive Bayes  0.425676   0.404991 0.433238 0.394231 0.596841
    4         RNN  0.408784   0.366700 0.407056 0.363636 0.603946
    4        LSTM  0.415541   0.270278 0.443789 0.335711 0.631360
    5 Naive Bayes  0.385135   0.367132 0.409611 0.368218 0.565063
    5         RNN  0.449324   0.421088 0.459188 0.419461 0.604361
    5     

In [3]:
summary_df = (fold_results_df.groupby('Model').agg(Accuracy_mean=('Accuracy', 'mean'), Accuracy_std=('Accuracy', 'std'), Precision_mean=('Precision', 'mean'), Precision_std=('Precision', 'std'), Recall_mean=('Recall', 'mean'), Recall_std=('Recall', 'std'), F1_mean=('F1', 'mean'), F1_std=('F1', 'std'), ROC_AUC_mean=('ROC-AUC', 'mean'), ROC_AUC_std=('ROC-AUC', 'std')).reset_index())
fold_results_df.to_csv(FOLD_RESULTS_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)
print(summary_df.to_string(index=False))


      Model  Accuracy_mean  Accuracy_std  Precision_mean  Precision_std  Recall_mean  Recall_std  F1_mean   F1_std  ROC_AUC_mean  ROC_AUC_std
       LSTM       0.398369      0.017876        0.358793       0.052180     0.410003    0.023662 0.361854 0.027551      0.589306     0.032651
Naive Bayes       0.407831      0.022150        0.400106       0.020116     0.437300    0.022803 0.382944 0.018399      0.586364     0.015878
        RNN       0.426743      0.014776        0.405696       0.022097     0.439073    0.024596 0.406122 0.023847      0.601456     0.007274


## Conclusion

Naive Bayes, RNN, and LSTM are supplementary exploratory models only. The RNN and LSTM use an artificial feature sequence for tabular comparison and do not imply natural temporal observations. These models are not replacements for the six primary models or the official sensitivity analysis.
